# SMS Spam Classification using SVM
Dataset: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset?resource=download

In [ ]:
!pip install wordcloud nltk scikit-learn lime

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from wordcloud import WordCloud
import re, string, nltk
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from lime.lime_text import LimeTextExplainer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score,classification_report,
confusion_matrix,ConfusionMatrixDisplay,
precision_score,recall_score,f1_score,roc_curve,auc)

nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


## Upload dataset (`spam.csv`) downloaded from Kaggle.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv("spam.csv", encoding="latin-1")

df = df[['v1', 'v2']]

df.columns = ['label', 'message']

In [ ]:
print(df.shape)
print(df.info())
print(df.isnull().sum())
print("Duplicates:",df.duplicated().sum())
print(df['label'].value_counts())


## Exploratory Data Analysis

In [ ]:
df['char_count']=df['message'].str.len()
df['word_count']=df['message'].str.split().str.len()

fig,ax=plt.subplots(1,2,figsize=(10,4))
sns.countplot(data=df,x='label',ax=ax[0])
sns.histplot(data=df,x='word_count',hue='label',bins=30,ax=ax[1])
plt.tight_layout()
plt.show()




In [ ]:
spam=' '.join(df[df.label=='spam'].message)
ham=' '.join(df[df.label=='ham'].message)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.imshow(WordCloud(background_color='white').generate(spam))
plt.axis('off'); plt.title("Spam")
plt.subplot(1,2,2)
plt.imshow(WordCloud(background_color='white').generate(ham))
plt.axis('off'); plt.title("Ham")
plt.show()

## Preprocessing

In [ ]:
# Load the list of English stopwords (e.g., "the", "is", "and")
stop_words = set(stopwords.words('english'))

# Create a WordNet lemmatizer object to convert words to their base form
lemm = WordNetLemmatizer()

# Define a function to clean and preprocess SMS text
def clean_text(text):

    # Convert all characters to lowercase
    text = text.lower()

    # Remove URLs (e.g., http://example.com or www.example.com)
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Remove email addresses (e.g., abc@gmail.com)
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove all numeric digits from the text
    text = re.sub(r'\d+', ' ', text)

    # Remove punctuation marks such as !, ?, ., ,, :, etc.
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Split the sentence into words,
    # remove stopwords,
    # and convert each word to its base (lemma) form
    tokens = [
        lemm.lemmatize(word)
        for word in text.split()
        if word not in stop_words
    ]

    # Join the cleaned words back into a single string
    return ' '.join(tokens)

# Remove duplicate SMS messages from the dataset
df = df.drop_duplicates()

# Apply the clean_text() function to every message
# and store the cleaned text in a new column named 'clean_message'
df['clean_message'] = df['message'].apply(clean_text)

# Convert categorical labels into numerical values
# ham  -> 0
# spam -> 1
df['label'] = df['label'].map({
    'ham': 0,
    'spam': 1
})

# Display the first five rows of the processed dataset
df.head()

## Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df['clean_message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)


## SVM Model

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        max_features=10000,
        ngram_range=(1,2),
        sublinear_tf=True
    )),
    ('svm', CalibratedClassifierCV(
        LinearSVC(
            class_weight='balanced',
            max_iter=10000
        ),
        cv=5
    ))
])

pipeline.fit(X_train,y_train)
pred=pipeline.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred))
print("Precision:",precision_score(y_test,pred))
print("Recall:",recall_score(y_test,pred))
print("F1:",f1_score(y_test,pred))
print(classification_report(y_test,pred))




## Hyperparameter Tuning

In [ ]:
param_grid = {
    'svm__estimator__C':[0.01,0.1,1,10,100]
}
grid=GridSearchCV(pipeline,param_grid,cv=5,n_jobs=-1)
grid.fit(X_train,y_train)

print("Best Parameters:",grid.best_params_)
print("Best CV Score:",grid.best_score_)

In [ ]:
model = grid.best_estimator_

In [ ]:
model.predict_proba(
    ["Congratulations you won a free prize"]
)

## Cross Validation

In [ ]:
scores=cross_val_score(grid.best_estimator_,df['clean_message'],df['label'],cv=5)
print(scores)
print("Mean Accuracy:",scores.mean())


## Prediction Function

In [ ]:
model=grid.best_estimator_

def predict_sms(text):

    cleaned = clean_text(text)

    pred = model.predict([cleaned])[0]

    probability = model.predict_proba([cleaned])[0]

    if pred == 1:
        return f"Spam ({probability[1]*100:.2f}% confidence)"
    else:
        return f"Ham ({probability[0]*100:.2f}% confidence)"

In [ ]:
def predict_probability(texts):

    probabilities = []

    for text in texts:

        prob = model.predict_proba([text])[0]

        probabilities.append(prob)

    return np.array(probabilities)

In [ ]:
class_names = ['Ham', 'Spam']

explainer = LimeTextExplainer(
    class_names=class_names
)

In [ ]:
def explain_sms(text):

    explanation = explainer.explain_instance(
        text,
        predict_probability,
        num_features=6
    )

    print(predict_sms(text))

    return explanation

In [ ]:
exp = explain_sms(
    "Congratulations! You have won a free iPhone. Click here now."
)

exp.show_in_notebook()

In [ ]:
exp.as_pyplot_figure()

In [ ]:
exp = explain_sms(
    "URGENT! You have won $1000 cash prize. Call now to claim your reward."
)

exp.show_in_notebook()

In [ ]:
exp = explain_sms(
    "Hey, are you coming to class tomorrow? Let me know."
)

exp.show_in_notebook()

In [ ]:
exp = explain_sms(
    "You have received a special offer from our company. Contact us for details."
)

exp.show_in_notebook()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("Precision:", precision_score(
    y_test,
    y_pred,
    pos_label=1
))

print("Recall:", recall_score(
    y_test,
    y_pred,
    pos_label=1
))

print("F1 Score:", f1_score(
    y_test,
    y_pred,
    pos_label=1
))